# Notebook 3 — Treinamento ML Raiz (XGBoost + outras arquiteturas)

**Objetivo**: Treinar modelos de ML clássico sobre o dataset com strings (sem embeddings pré-computados),
replicando o protocolo do pipeline antigo para comparação futura com LLMs.

**Protocolo**:
- N_SEEDS = 20 (mesma quantidade que o LLM usará — instrução Eduardo)
- Avaliação no test set (200 amostras)
- Métricas: RMSE, R², MAE por seed
- Modelos: XGBoost (HP do Optuna), LightGBM, Random Forest

**Seção 2 (opcional)**: Pipeline de embeddings — embeda as colunas de texto com
`paraphrase-multilingual-MiniLM-L12-v2` (mesmo modelo do pipeline antigo)
caso queiramos adicionar embeddings às features tabulares.

In [18]:
!pip install xgboost lightgbm
!pip install -U sentence-transformers


In [17]:
import gc
import json
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.stats import wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# ── Caminhos ──────────────────────────────────────────────────────────────────
INPUT_DIR   = Path('data_input')
HP_DIR      = Path('hyperparams')
PREDS_DIR   = Path('preds')
METRICS_DIR = Path('metrics') / 'ML_models'
MODELS_DIR  = Path('models')

for d in [PREDS_DIR, METRICS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Parâmetros ────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_SEEDS     = 20
VAL_SIZE    = 0.10
N_BOOST     = 3000
EARLY_STOP  = 50

# ── Colunas ───────────────────────────────────────────────────────────────────
ID_ATRAC = 'IDAtracacao'
PORT_COL = 'Porto Atracação'

# TODOS os 6 targets — nenhum pode ser feature
ALL_TARGETS = [
    'TEstadia', 'TEsperaAtracacao', 'TAtracado',
    'TEsperaInicioOp', 'TOperacao', 'TEsperaDesatracacao',
]
TARGET_COLS = ['TOperacao', 'TAtracado']  # targets preditos neste paper

TEXT_COLS = [
    'Origem', 'Destino',
    'Grupo de Mercadoria', 'Grupo Mercadoria Conteinerizada',
]
CAT_COLS = [
    'Porto Atracação', 'Complexo Portuário', 'Tipo da Autoridade Portuária',
    'Tipo de Operação', 'Tipo de Navegação da Atracação',
    'Município', 'UF', 'SGUF', 'Região Geográfica', 'Região Hidrográfica',
    'Instalação Portuária em Rio', 'Natureza da Carga',
    'Percurso Transporte Interiores', 'STNaturezaCarga',
    'Carga Geral Acondicionamento',
]

print('Configuração OK')
print(f'Targets excluídos das features: {ALL_TARGETS}')


Configuração OK
Targets excluídos das features: ['TEstadia', 'TEsperaAtracacao', 'TAtracado', 'TEsperaInicioOp', 'TOperacao', 'TEsperaDesatracacao']


## 1. Carregar dados

In [18]:
print('Carregando train_strings.parquet...')
df_train = pq.read_table(str(INPUT_DIR / 'train_strings.parquet')).to_pandas()
print(f'  train: {df_train.shape}')

print('Carregando test_strings.parquet...')
df_test = pq.read_table(str(INPUT_DIR / 'test_strings.parquet')).to_pandas()
print(f'  test : {df_test.shape}')

ohe_cols = [c for c in df_train.columns if c.startswith('ohe_')]

# Exclui TODOS os 6 targets, textos, categóricas, OHE e ID
num_cols = [
    c for c in df_train.columns
    if c not in ALL_TARGETS + TEXT_COLS + CAT_COLS + ohe_cols + [ID_ATRAC]
    and '_embedding_' not in c
]

print(f'\nColunas categóricas : {len(CAT_COLS)}')
print(f'Colunas numéricas   : {len(num_cols)}')
print(f'Colunas OHE+SUM     : {len(ohe_cols)}')
print(f'Colunas texto       : {len(TEXT_COLS)}')
print(f'Targets (todos 6)   : {ALL_TARGETS}')
print(f'Targets preditos    : {TARGET_COLS}')
print(f'\nNum cols: {num_cols}')


Carregando train_strings.parquet...
  train: (391460, 77)
Carregando test_strings.parquet...
  test : (200, 77)

Colunas categóricas : 15
Colunas numéricas   : 21
Colunas OHE+SUM     : 34
Colunas texto       : 4
Targets (todos 6)   : ['TEstadia', 'TEsperaAtracacao', 'TAtracado', 'TEsperaInicioOp', 'TOperacao', 'TEsperaDesatracacao']
Targets preditos    : ['TOperacao', 'TAtracado']

Num cols: ['Nacionalidade do Armador', 'lon', 'lat', 'mes_sin', 'mes_cos', 'Ano', 'Mes_num', 'DiaSemana', 'VLPesoCargaBruta', 'TEU', 'QTCarga', 'VLPesoCargaConteinerizada_total', 'valor_mov_regiao_total', 'valor_mov_rio_total', 'valor_mov_hidrovia_total', 'valor_mov_total_hidrografia', 'qt_registros_carga', 'qt_naturezas_carga', 'qt_distinct_natureza_carga', 'qt_distinct_percurso_interiores', 'qt_distinct_carga_geral_acond']


## 2. Pipeline de Embeddings (opcional)

Replica a lógica do `embeddings.py` do pipeline antigo, adaptada para nível de atracação.
Usa o mesmo modelo: `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensões).

**Execute esta seção apenas se quiser adicionar embeddings às features tabulares.**
A flag `USE_EMBEDDINGS` controla se os embeddings são incluídos no treinamento.

In [19]:
USE_EMBEDDINGS = True  # Mude para True para incluir embeddings nas features

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMB_DIM    = 384
BATCH_SIZE = 128

print(f'USE_EMBEDDINGS = {USE_EMBEDDINGS}')

USE_EMBEDDINGS = True


In [20]:
def embed_text_col(df_train, df_test, text_col, model, emb_dim=EMB_DIM):
    """
    Embeda uma coluna de texto para treino e teste.
    Deduplica textos (igual ao embeddings.py antigo) e faz lookup.
    Retorna arrays (n_train, emb_dim) e (n_test, emb_dim).
    """
    texts_train = df_train[text_col].fillna('').astype(str).tolist()
    texts_test  = df_test[text_col].fillna('').astype(str).tolist()

    unique_texts = sorted(set(texts_train) | set(texts_test))
    print(f'  {text_col}: {len(unique_texts)} textos únicos')

    raw = model.encode(
        unique_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype('float32')

    text_to_emb = {t: raw[i] for i, t in enumerate(unique_texts)}
    fallback = np.zeros(emb_dim, dtype='float32')

    emb_train = np.stack([text_to_emb.get(t, fallback) for t in texts_train])
    emb_test  = np.stack([text_to_emb.get(t, fallback) for t in texts_test])

    del raw, text_to_emb
    gc.collect()
    return emb_train, emb_test


emb_train_dict = {}  # text_col -> np.array (n_train, 384)
emb_test_dict  = {}  # text_col -> np.array (n_test, 384)

if USE_EMBEDDINGS:
    try:
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError:
        device = 'cpu'

    from sentence_transformers import SentenceTransformer
    print(f'Carregando modelo {MODEL_NAME} (device={device})...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    model.max_seq_length = 256

    for col in TEXT_COLS:
        etr, ete = embed_text_col(df_train, df_test, col, model)
        emb_train_dict[col] = etr
        emb_test_dict[col]  = ete

    del model
    gc.collect()
    print(f'\nEmbeddings gerados: {len(TEXT_COLS)} colunas × {EMB_DIM} dims')
else:
    print('Embeddings desativados (USE_EMBEDDINGS=False)')

Carregando modelo sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (device=cuda)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3090.01it/s]


  Origem: 1350 textos únicos


Batches: 100%|██████████| 11/11 [00:00<00:00, 61.85it/s]


  Destino: 1709 textos únicos


...

  Grupo de Mercadoria: 73 textos únicos


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.57it/s]


  Grupo Mercadoria Conteinerizada: 92 textos únicos


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.50it/s]



Embeddings gerados: 4 colunas × 384 dims


## 3. Preparar features

- Categóricas: OrdinalEncoder (fitado no treino)
- `Porto Atracação`: median target encoding por target (igual ao `train_final.py`)
- Numéricas + OHE+SUM: usadas diretamente
- Embeddings: concatenados se `USE_EMBEDDINGS=True`

In [21]:
# Colunas categóricas sem Porto Atracação (porto entra por median encoding por target)
cat_cols_no_port = [c for c in CAT_COLS if c != PORT_COL]

# Feature columns base (sem porto, sem text, sem targets, sem ID)
feat_base_cols = cat_cols_no_port + num_cols + ohe_cols

# OrdinalEncoder nas categóricas (fitado no treino)
cats_in_train = [c for c in cat_cols_no_port if c in df_train.columns]
enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    encoded_missing_value=-1,
)
enc.fit(df_train[cats_in_train].astype(str))

def build_X(df, emb_dict=None):
    """Constrói matriz de features numpy para um DataFrame."""
    parts = []

    # Categóricas codificadas (sem porto)
    if cats_in_train:
        X_cat = enc.transform(df[cats_in_train].astype(str)).astype(np.float32)
        parts.append(X_cat)

    # Numéricas + OHE+SUM
    num_ohe = [c for c in num_cols + ohe_cols if c in df.columns]
    X_num = df[num_ohe].to_numpy(dtype=np.float32)
    parts.append(X_num)

    # Embeddings (opcionais)
    if emb_dict:
        for col in TEXT_COLS:
            if col in emb_dict:
                parts.append(emb_dict[col])

    return np.concatenate(parts, axis=1)


X_train_base = build_X(df_train, emb_train_dict if USE_EMBEDDINGS else None)
X_test_base  = build_X(df_test,  emb_test_dict  if USE_EMBEDDINGS else None)

train_port = df_train[PORT_COL].to_numpy()
test_port  = df_test[PORT_COL].to_numpy()

# Split interno treino/validação para early stopping
core_idx, val_idx = train_test_split(
    np.arange(len(df_train)), test_size=VAL_SIZE, random_state=RANDOM_SEED
)
X_core = X_train_base[core_idx]
X_val  = X_train_base[val_idx]
port_core = train_port[core_idx]
port_val  = train_port[val_idx]

n_feats = X_train_base.shape[1]
print(f'Features base      : {n_feats}')
print(f'  - categóricas    : {len(cats_in_train)}')
print(f'  - numéricas+OHE  : {len([c for c in num_cols + ohe_cols if c in df_train.columns])}')
if USE_EMBEDDINGS:
    print(f'  - embeddings     : {len(TEXT_COLS) * EMB_DIM}')
print(f'Treino core/val    : {len(core_idx):,} / {len(val_idx):,}')
print(f'Teste              : {len(df_test):,}')

Features base      : 1605
  - categóricas    : 14
  - numéricas+OHE  : 55
  - embeddings     : 1536
Treino core/val    : 352,314 / 39,146
Teste              : 200


In [22]:
def port_median_encode(port_arr, y_arr, port_test_arr, global_med):
    """Median target encoding para Porto Atracação (igual ao train_final.py)."""
    medians = pd.Series(y_arr, index=port_arr).groupby(level=0).median().to_dict()
    encode  = lambda arr: pd.Series(arr).map(medians).fillna(global_med).to_numpy(dtype=np.float32)
    return encode(port_arr), encode(port_test_arr)


In [23]:
import tempfile, os
from sklearn.preprocessing import StandardScaler

_MMAP_CHUNK = 50_000   # linhas por vez ao gravar/ler memmap

def build_mmap(X_base, port_enc_1d, tag):
    """
    Grava (X_base | port_enc) num arquivo memory-mapped em chunks.
    Nunca duplica a matriz inteira na RAM — o SO faz paging sob demanda.
    Retorna (np.memmap, path_str).
    """
    n, p = X_base.shape
    path = str(Path(tempfile.gettempdir()) / f'antaq_{tag}.dat')
    mm   = np.memmap(path, dtype='float32', mode='w+', shape=(n, p + 1))
    for s in range(0, n, _MMAP_CHUNK):
        e = min(s + _MMAP_CHUNK, n)
        mm[s:e, :-1] = X_base[s:e]
        mm[s:e, -1]  = port_enc_1d[s:e]
    mm.flush()
    return mm, path

def scaler_from_mmap(mm, batch=_MMAP_CHUNK):
    """Fita StandardScaler em batches a partir de um memmap."""
    sc = StandardScaler()
    for s in range(0, mm.shape[0], batch):
        sc.partial_fit(mm[s:s+batch])
    return sc

def build_scaled_mmap(mm, sc, tag):
    """Grava memmap normalizado no disco em batches — sem alocar 3 GB na RAM."""
    path = str(Path(tempfile.gettempdir()) / f'antaq_{tag}_scaled.dat')
    sm   = np.memmap(path, dtype='float32', mode='w+', shape=mm.shape)
    for s in range(0, mm.shape[0], _MMAP_CHUNK):
        e = min(s + _MMAP_CHUNK, mm.shape[0])
        sm[s:e] = sc.transform(mm[s:e])
    sm.flush()
    return sm, path

def rm_mmap(*paths):
    for p in paths:
        try:
            os.unlink(p)
        except Exception:
            pass

print('Helpers de memmap prontos.')


Helpers de memmap prontos.


## 4. XGBoost — HP do Optuna (piso, por target)

Usa os `best_params_target_*.json` encontrados pelo Optuna no pipeline antigo.
Porto Atracação: median target encoding (fitted on core, applied to val e test).

In [14]:
import xgboost as xgb

def detect_device():
    try:
        xgb.XGBRegressor(device='cuda', n_estimators=1).fit(
            np.array([[1.0]]), np.array([0.0])
        )
        return 'cuda'
    except Exception:
        return 'cpu'


def port_median_encode(port_arr, y_arr, port_test_arr, global_med):
    """Median target encoding para Porto Atracação (igual ao train_final.py)."""
    medians = pd.Series(y_arr, index=port_arr).groupby(level=0).median().to_dict()
    encode  = lambda arr: pd.Series(arr).map(medians).fillna(global_med).to_numpy(dtype=np.float32)
    return encode(port_arr), encode(port_test_arr)


def train_xgb_one(X_tr, y_tr, X_val, y_val, X_te, y_te, params, seed, device):
    p = dict(params)
    p.update({
        'objective': 'reg:squarederror',
        'device': device,
        'tree_method': 'hist',
        'eval_metric': 'rmse',
        'seed': seed,
        'n_jobs': -1,
    })
    if device == 'cuda' and 'max_bin' not in p:
        p['max_bin'] = 256
    p.pop('n_estimators', None)
    max_bin = int(p.get('max_bin', 256))

    dtr  = xgb.QuantileDMatrix(X_tr,  label=y_tr,  max_bin=max_bin)
    dval = xgb.QuantileDMatrix(X_val, label=y_val, ref=dtr, max_bin=max_bin)
    dte  = xgb.QuantileDMatrix(X_te,  ref=dtr, max_bin=max_bin)

    booster = xgb.train(
        p, dtr, num_boost_round=N_BOOST,
        evals=[(dval, 'val')],
        early_stopping_rounds=EARLY_STOP,
        verbose_eval=False,
    )
    pred = booster.predict(dte, iteration_range=(0, booster.best_iteration + 1))
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),
        'mae':  float(mean_absolute_error(y_te, pred)),
        'best_iter': int(booster.best_iteration),
        'pred': pred,
        'model_obj': booster,
    }


device = detect_device()
print(f'Device XGBoost: {device}')

Device XGBoost: cuda


In [15]:
results_xgb = []
preds_xgb   = {}

for target in TARGET_COLS:
    hp_path = HP_DIR / f'best_params_{target}.json'
    with open(hp_path) as f:
        params = json.load(f)

    y_core = df_train[target].to_numpy(dtype=np.float32)[core_idx]
    y_val  = df_train[target].to_numpy(dtype=np.float32)[val_idx]
    y_te   = df_test[target].to_numpy(dtype=np.float32)

    global_med = float(np.median(y_core))
    port_enc_core, port_enc_test = port_median_encode(
        port_core, y_core, test_port, global_med
    )
    port_enc_val = pd.Series(port_val).map(
        pd.Series(y_core, index=port_core).groupby(level=0).median().to_dict()
    ).fillna(global_med).to_numpy(dtype=np.float32)

    X_tr = np.column_stack([X_core, port_enc_core])
    X_vl = np.column_stack([X_val,  port_enc_val])
    X_te = np.column_stack([X_test_base, port_enc_test])

    seed_preds = []
    print(f'\n── {target} ──')
    for seed in range(N_SEEDS):
        try:
            row = train_xgb_one(X_tr, y_core, X_vl, y_val, X_te, y_te, params, seed, device)
        except Exception as e:
            if 'out of memory' in str(e).lower() and device == 'cuda':
                row = train_xgb_one(X_tr, y_core, X_vl, y_val, X_te, y_te, params, seed, 'cpu')
            else:
                raise

        pred    = row.pop('pred')
        booster = row.pop('model_obj')

        np.save(PREDS_DIR / f'xgboost_{target}_seed{seed}.npy', pred)
        booster.save_model(str(MODELS_DIR / f'xgboost_{target}_seed{seed}.json'))

        seed_preds.append(pred)
        row.update({'model': 'xgboost', 'target': target, 'seed': seed})
        results_xgb.append(row)
        print(f'  seed={seed:2d}  RMSE={row["rmse"]:.3f}  R²={row["r2"]:.4f}  iter={row["best_iter"]}')
        gc.collect()
        

    preds_xgb[target] = np.stack(seed_preds)

df_xgb = pd.DataFrame(results_xgb)
df_xgb.to_csv(METRICS_DIR / 'results_xgboost.csv', index=False)
print('\nResultados XGBoost salvos.')
df_xgb.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)


── TOperacao ──
  seed= 0  RMSE=19.207  R²=0.7216  iter=1421
  seed= 1  RMSE=18.799  R²=0.7334  iter=1306
  seed= 2  RMSE=19.065  R²=0.7257  iter=1024
  seed= 3  RMSE=18.962  R²=0.7287  iter=1871
  seed= 4  RMSE=18.978  R²=0.7282  iter=1384
  seed= 5  RMSE=19.267  R²=0.7199  iter=1493
  seed= 6  RMSE=19.177  R²=0.7225  iter=1225
  seed= 7  RMSE=18.783  R²=0.7338  iter=1170
  seed= 8  RMSE=19.066  R²=0.7257  iter=1225
  seed= 9  RMSE=18.962  R²=0.7287  iter=1245
  seed=10  RMSE=19.400  R²=0.7160  iter=952
  seed=11  RMSE=19.093  R²=0.7249  iter=1506
  seed=12  RMSE=19.275  R²=0.7197  iter=930
  seed=13  RMSE=19.173  R²=0.7226  iter=1011
  seed=14  RMSE=19.203  R²=0.7218  iter=1567
  seed=15  RMSE=19.301  R²=0.7189  iter=818
  seed=16  RMSE=18.888  R²=0.7308  iter=1572
  seed=17  RMSE=19.168  R²=0.7228  iter=1800
  seed=18  RMSE=18.949  R²=0.7291  iter=1459
  seed=19  RMSE=19.263  R²=0.7200  iter=882

── TAtracado ──
  seed= 0  RMSE=20.955  R²=0.7283  iter=565
  seed= 1  RMSE=20.812  R²

rmse              r2              mae        
              mean     std    mean     std     mean     std
target                                                     
TAtracado  20.9571  0.1657  0.7283  0.0043  10.7223  0.0950
TOperacao  19.0991  0.1729  0.7248  0.0050   7.7503  0.0877

## 5. SVM (LinearSVR)

SVR clássico é O(n²) — inviável com 489k amostras. Usamos `LinearSVR` que escala linearmente.
Sem hiper-otimização: parâmetros padrão sensatos (patamar).

In [24]:
from sklearn.linear_model import SGDRegressor

_SGD_BATCH  = 10_000
_SGD_EPOCHS = 10      # épocas com shuffle manual

In [25]:
from sklearn.linear_model import SGDRegressor

_SGD_BATCH  = 10_000
_SGD_EPOCHS = 10      # épocas com shuffle manual

def train_svm_one(mm_scaled, y_tr, X_te_scaled, y_te, seed):
    """
    LinearSVR aproximado via SGDRegressor(epsilon_insensitive).
    Treina com partial_fit em batches — nunca carrega a matriz inteira.
    """
    n   = len(y_tr)
    rng = np.random.default_rng(seed)
    model = SGDRegressor(
        loss='epsilon_insensitive', epsilon=0.0,
        penalty='l2', alpha=1e-4,
        max_iter=1, tol=None, warm_start=False,
        random_state=int(rng.integers(1 << 31)),
    )
    for _ in range(_SGD_EPOCHS):
        order = rng.permutation(n)
        for s in range(0, n, _SGD_BATCH):
            idx = order[s:s + _SGD_BATCH]
            model.partial_fit(mm_scaled[idx], y_tr[idx])
    pred = model.predict(X_te_scaled)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),
        'mae':  float(mean_absolute_error(y_te, pred)),
        'pred': pred,
        'model_obj': model,
    }


results_svm = []
preds_svm   = {}

for target in TARGET_COLS:
    y_tr_full = df_train[target].to_numpy(dtype=np.float32)
    y_te      = df_test[target].to_numpy(dtype=np.float32)
    global_med = float(np.median(y_tr_full))

    port_enc_tr, port_enc_te = port_median_encode(
        train_port, y_tr_full, test_port, global_med
    )

    print(f'\n── {target}: gravando memmap...')
    mm, mm_path = build_mmap(X_train_base, port_enc_tr, f'svm_{target}')
    sc          = scaler_from_mmap(mm)
    sm, sm_path = build_scaled_mmap(mm, sc, f'svm_{target}')
    del mm
    gc.collect()

    X_te_m      = np.column_stack([X_test_base, port_enc_te])   # 200 linhas — OK
    X_te_scaled = sc.transform(X_te_m)

    seed_preds = []
    for seed in range(N_SEEDS):
        row       = train_svm_one(sm, y_tr_full, X_te_scaled, y_te, seed)
        pred      = row.pop('pred')
        model_obj = row.pop('model_obj')

        np.save(PREDS_DIR / f'svm_{target}_seed{seed}.npy', pred)
        joblib.dump((sc, model_obj), MODELS_DIR / f'svm_{target}_seed{seed}.pkl')

        seed_preds.append(pred)
        row.update({'model': 'svm', 'target': target, 'seed': seed})
        results_svm.append(row)
        print(f'  seed={seed:2d}  RMSE={row["rmse"]:.3f}  R²={row["r2"]:.4f}')

    preds_svm[target] = np.stack(seed_preds)
    rm_mmap(mm_path, sm_path)
    gc.collect()

df_svm = pd.DataFrame(results_svm)
df_svm.to_csv(METRICS_DIR / 'results_svm.csv', index=False)
print('\nResultados SVM salvos.')
df_svm.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)



── TOperacao: gravando memmap...
  seed= 0  RMSE=26.344  R²=0.4764
  seed= 1  RMSE=26.295  R²=0.4783
  seed= 2  RMSE=26.236  R²=0.4806
  seed= 3  RMSE=26.224  R²=0.4811
  seed= 4  RMSE=26.345  R²=0.4763
  seed= 5  RMSE=26.181  R²=0.4828
  seed= 6  RMSE=26.375  R²=0.4751
  seed= 7  RMSE=26.179  R²=0.4829
  seed= 8  RMSE=26.356  R²=0.4759
  seed= 9  RMSE=26.292  R²=0.4784
  seed=10  RMSE=26.284  R²=0.4788
  seed=11  RMSE=26.315  R²=0.4775
  seed=12  RMSE=26.350  R²=0.4761
  seed=13  RMSE=26.279  R²=0.4789
  seed=14  RMSE=26.331  R²=0.4769
  seed=15  RMSE=26.349  R²=0.4762
  seed=16  RMSE=26.421  R²=0.4733
  seed=17  RMSE=26.204  R²=0.4819
  seed=18  RMSE=26.341  R²=0.4765
  seed=19  RMSE=26.254  R²=0.4799

── TAtracado: gravando memmap...
  seed= 0  RMSE=28.672  R²=0.4914
  seed= 1  RMSE=28.720  R²=0.4897
  seed= 2  RMSE=28.712  R²=0.4900
  seed= 3  RMSE=28.692  R²=0.4907
  seed= 4  RMSE=28.774  R²=0.4878
  seed= 5  RMSE=28.568  R²=0.4951
  seed= 6  RMSE=28.773  R²=0.4878
  seed= 7  RMS

rmse              r2              mae        
              mean     std    mean     std     mean     std
target                                                     
TAtracado  28.6754  0.0831  0.4913  0.0029  13.7324  0.1133
TOperacao  26.2977  0.0672  0.4782  0.0027  10.3524  0.0685

## 6. KNN (K-Nearest Neighbors Regressor)

Sem hiper-otimização. k=10 como patamar razoável.

In [10]:
from sklearn.neighbors import KNeighborsRegressor

def train_knn_one(X_tr_scaled, y_tr, X_te_scaled, y_te):
    """
    KNN com todos os dados. X_tr_scaled já é memmap normalizado.
    sklearn copia internamente 1× (inevitável) — mas sem o column_stack extra.
    """
    model = KNeighborsRegressor(n_neighbors=10, n_jobs=-1)
    model.fit(X_tr_scaled, y_tr)       # 1× cópia interna para BallTree
    pred = model.predict(X_te_scaled)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),
        'mae':  float(mean_absolute_error(y_te, pred)),
        'pred': pred,
        'model_obj': model,
    }


results_knn = []
preds_knn   = {}

for target in TARGET_COLS:
    y_tr_full = df_train[target].to_numpy(dtype=np.float32)
    y_te      = df_test[target].to_numpy(dtype=np.float32)
    global_med = float(np.median(y_tr_full))

    port_enc_tr, port_enc_te = port_median_encode(
        train_port, y_tr_full, test_port, global_med
    )

    print(f'\n── {target}: gravando memmap...')
    mm, mm_path = build_mmap(X_train_base, port_enc_tr, f'knn_{target}')
    sc          = scaler_from_mmap(mm)
    sm, sm_path = build_scaled_mmap(mm, sc, f'knn_{target}')
    del mm
    gc.collect()

    X_te_m      = np.column_stack([X_test_base, port_enc_te])
    X_te_scaled = sc.transform(X_te_m)

    # KNN é determinístico dado X_tr — 20 seeds produzem o mesmo modelo.
    # Treinamos 1 vez e replicamos as predições para protocolo Wilcoxon.
    print('  Treinando KNN (todos os dados)...')
    row       = train_knn_one(sm, y_tr_full, X_te_scaled, y_te)
    pred_base = row.pop('pred')
    model_obj = row.pop('model_obj')

    seed_preds = []
    for seed in range(N_SEEDS):
        np.save(PREDS_DIR / f'knn_{target}_seed{seed}.npy', pred_base)
        seed_preds.append(pred_base)
        r = {**row, 'model': 'knn', 'target': target, 'seed': seed}
        results_knn.append(r)
        print(f'  seed={seed:2d}  RMSE={r["rmse"]:.3f}  R²={r["r2"]:.4f}')

    joblib.dump((sc, model_obj), MODELS_DIR / f'knn_{target}.pkl')
    preds_knn[target] = np.stack(seed_preds)

    del model_obj, sm
    rm_mmap(mm_path, sm_path)
    gc.collect()

df_knn = pd.DataFrame(results_knn)
df_knn.to_csv(METRICS_DIR / 'results_knn.csv', index=False)
print('\nResultados KNN salvos.')
df_knn.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)



── TOperacao: gravando memmap...
  Treinando KNN (todos os dados)...
  seed= 0  RMSE=22.171  R²=0.6291
  seed= 1  RMSE=22.171  R²=0.6291
  seed= 2  RMSE=22.171  R²=0.6291
  seed= 3  RMSE=22.171  R²=0.6291
  seed= 4  RMSE=22.171  R²=0.6291
  seed= 5  RMSE=22.171  R²=0.6291
  seed= 6  RMSE=22.171  R²=0.6291
  seed= 7  RMSE=22.171  R²=0.6291
  seed= 8  RMSE=22.171  R²=0.6291
  seed= 9  RMSE=22.171  R²=0.6291
  seed=10  RMSE=22.171  R²=0.6291
  seed=11  RMSE=22.171  R²=0.6291
  seed=12  RMSE=22.171  R²=0.6291
  seed=13  RMSE=22.171  R²=0.6291
  seed=14  RMSE=22.171  R²=0.6291
  seed=15  RMSE=22.171  R²=0.6291
  seed=16  RMSE=22.171  R²=0.6291
  seed=17  RMSE=22.171  R²=0.6291
  seed=18  RMSE=22.171  R²=0.6291
  seed=19  RMSE=22.171  R²=0.6291

── TAtracado: gravando memmap...
  Treinando KNN (todos os dados)...
  seed= 0  RMSE=23.125  R²=0.6692
  seed= 1  RMSE=23.125  R²=0.6692
  seed= 2  RMSE=23.125  R²=0.6692
  seed= 3  RMSE=23.125  R²=0.6692
  seed= 4  RMSE=23.125  R²=0.6692
  seed= 5 

rmse           r2           mae     
              mean  std    mean  std     mean  std
target                                            
TAtracado  23.1247  0.0  0.6692  0.0  12.1293  0.0
TOperacao  22.1709  0.0  0.6291  0.0   9.6193  0.0

## 7. Naive Bayes → Linear Regression (patamar)

**Nota**: Naive Bayes é um algoritmo de classificação — não se aplica a regressão.
Usamos **Regressão Linear** como equivalente "naive" para regressão: modelo mais simples possível,
serve como patamar inferior de referência.

In [13]:
def train_lr_one(mm_scaled, y_tr, X_te_scaled, y_te, seed):
    """
    OLS via equações normais computadas em chunks do memmap.
    Bootstrap por seed (amostragem COM reposição, mesmo tamanho) — gera
    variância real entre seeds sem subsampling.
    Memória: O(p²) = ~20 MB para XtX. Nunca carrega a matriz inteira.
    """
    rng = np.random.default_rng(seed)
    n, p = mm_scaled.shape

    # Bootstrap: n amostras COM reposição
    boot_idx = rng.integers(0, n, size=n)

    XtX = np.zeros((p, p), dtype=np.float64)
    Xty = np.zeros(p,      dtype=np.float64)

    for s in range(0, n, _SGD_BATCH):
        idx = boot_idx[s:s + _SGD_BATCH]
        Xb  = mm_scaled[idx].astype(np.float64)
        yb  = y_tr[idx].astype(np.float64)
        XtX += Xb.T @ Xb
        Xty += Xb.T @ yb
        del Xb, yb

    # Regularização mínima para estabilidade numérica
    XtX[np.arange(p), np.arange(p)] += 1e-8
    w    = np.linalg.solve(XtX, Xty)
    pred = (X_te_scaled.astype(np.float64) @ w).astype(np.float32)

    return {
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),
        'mae':  float(mean_absolute_error(y_te, pred)),
        'pred': pred,
    }


results_lr = []
preds_lr   = {}

for target in TARGET_COLS:
    y_tr_full = df_train[target].to_numpy(dtype=np.float32)
    y_te      = df_test[target].to_numpy(dtype=np.float32)
    global_med = float(np.median(y_tr_full))

    port_enc_tr, port_enc_te = port_median_encode(
        train_port, y_tr_full, test_port, global_med
    )

    print(f'\n── {target}: gravando memmap...')
    mm, mm_path = build_mmap(X_train_base, port_enc_tr, f'lr_{target}')
    sc          = scaler_from_mmap(mm)
    sm, sm_path = build_scaled_mmap(mm, sc, f'lr_{target}')
    del mm
    gc.collect()

    X_te_m      = np.column_stack([X_test_base, port_enc_te])
    X_te_scaled = sc.transform(X_te_m)

    seed_preds = []
    for seed in range(N_SEEDS):
        row  = train_lr_one(sm, y_tr_full, X_te_scaled, y_te, seed)
        pred = row.pop('pred')

        np.save(PREDS_DIR / f'linear_regression_{target}_seed{seed}.npy', pred)

        seed_preds.append(pred)
        row.update({'model': 'linear_regression', 'target': target, 'seed': seed})
        results_lr.append(row)
        print(f'  seed={seed:2d}  RMSE={row["rmse"]:.3f}  R²={row["r2"]:.4f}')

    preds_lr[target] = np.stack(seed_preds)
    rm_mmap(mm_path, sm_path)
    gc.collect()

df_lr = pd.DataFrame(results_lr)
df_lr.to_csv(METRICS_DIR / 'results_linear_regression.csv', index=False)
print('\nResultados Linear Regression salvos.')
df_lr.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)



── TOperacao: gravando memmap...
  seed= 0  RMSE=24.989  R²=0.5289
  seed= 1  RMSE=25.216  R²=0.5202
  seed= 2  RMSE=25.247  R²=0.5190
  seed= 3  RMSE=25.707  R²=0.5014
  seed= 4  RMSE=25.334  R²=0.5158
  seed= 5  RMSE=25.208  R²=0.5206
  seed= 6  RMSE=25.175  R²=0.5218
  seed= 7  RMSE=24.935  R²=0.5309
  seed= 8  RMSE=24.868  R²=0.5334
  seed= 9  RMSE=25.110  R²=0.5243
  seed=10  RMSE=25.218  R²=0.5202
  seed=11  RMSE=25.117  R²=0.5240
  seed=12  RMSE=24.846  R²=0.5342
  seed=13  RMSE=24.991  R²=0.5288
  seed=14  RMSE=25.606  R²=0.5053
  seed=15  RMSE=25.078  R²=0.5255
  seed=16  RMSE=25.206  R²=0.5206
  seed=17  RMSE=24.980  R²=0.5292
  seed=18  RMSE=25.169  R²=0.5220
  seed=19  RMSE=25.048  R²=0.5266

── TAtracado: gravando memmap...
  seed= 0  RMSE=27.518  R²=0.5315
  seed= 1  RMSE=27.903  R²=0.5183
  seed= 2  RMSE=27.834  R²=0.5207
  seed= 3  RMSE=28.273  R²=0.5055
  seed= 4  RMSE=27.894  R²=0.5186
  seed= 5  RMSE=27.687  R²=0.5257
  seed= 6  RMSE=27.608  R²=0.5285
  seed= 7  RMS

rmse              r2              mae        
              mean     std    mean     std     mean     std
target                                                     
TAtracado  27.7212  0.2207  0.5246  0.0076  15.3884  0.1295
TOperacao  25.1524  0.2170  0.5226  0.0083  11.8138  0.1334

## 8. Rede Neural (MLP)

`MLPRegressor` do scikit-learn. Arquitetura simples como patamar: 2 camadas ocultas (256, 128).
Sem hiper-otimização.

In [16]:
import torch                                                                                                                                                                            
import torch.nn as nn
                                                                                                                                                                                        
_torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')                                                                                                            
print(f'MLP device: {_torch_device}')
                                                                                                                                                                                        
                                                                                                                                                                                    
class _MLP(nn.Module):                                                                                                                                                                  
    def __init__(self, n_in):                                                                                                                                                         
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.1),                                                                                                      
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128),                                                                                                                        
            nn.Linear(128, 1),                                                                                                                                                          
        )                                                                                                                                                                               
    def forward(self, x):                                                                                                                                                             
        return self.net(x).squeeze(-1)                                                                                                                                                  
                                                                                                                                                                                    
                                                                                                                                                                                        
def train_mlp_one(sm_scaled, y_tr, X_te_scaled, y_te, seed):
    torch.manual_seed(seed)                                                                                                                                                             
    if _torch_device.type == 'cuda':                                                                                                                                                  
        torch.cuda.manual_seed(seed)

    n, p   = sm_scaled.shape                                                                                                                                                            
    n_val  = max(1, int(0.1 * n))
    n_tr   = n - n_val                                                                                                                                                                  
    BATCH  = 2048                                                                                                                                                                     

    # Val fixo nas últimas n_val linhas (lê do memmap 1×)                                                                                                                               
    X_v  = torch.from_numpy(sm_scaled[n_tr:].copy()).to(_torch_device)
    y_v  = torch.from_numpy(y_tr[n_tr:]).to(_torch_device)                                                                                                                              
    X_te = torch.from_numpy(X_te_scaled).to(_torch_device)                                                                                                                              

    model   = _MLP(p).to(_torch_device)                                                                                                                                                 
    opt     = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)                                                                                                        
    loss_fn = nn.MSELoss()                                                                                                                                                              

    rng = np.random.default_rng(seed)                                                                                                                                                   
    best_val, best_state, no_imp = float('inf'), None, 0                                                                                                                              
                                                                                                                                                                                        
    for _ in range(300):                                                                                                                                                              
        model.train()
        order = rng.permutation(n_tr)
        for s in range(0, n_tr, BATCH):                                                                                                                                                 
            idx = order[s:s + BATCH]
            xb  = torch.from_numpy(sm_scaled[idx].copy()).to(_torch_device)                                                                                                             
            yb  = torch.from_numpy(y_tr[idx]).to(_torch_device)                                                                                                                         
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()                                                                                                                                           
            opt.step()                                                                                                                                                                
            del xb, yb
                                                                                                                                                                                        
        model.eval()
        with torch.no_grad():                                                                                                                                                           
            v_loss = loss_fn(model(X_v), y_v).item()                                                                                                                                  

        if v_loss < best_val - 1e-4:                                                                                                                                                    
            best_val   = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}                                                                                                    
            no_imp     = 0                                                                                                                                                            
        else:
            no_imp += 1                                                                                                                                                                 
            if no_imp >= 20:
                break                                                                                                                                                                   
                                                                                                                                                                                    
    model.load_state_dict({k: v.to(_torch_device) for k, v in best_state.items()})                                                                                                      
    model.eval()
    with torch.no_grad():                                                                                                                                                               
        pred = model(X_te).cpu().numpy()                                                                                                                                              

    del model, X_v, y_v, X_te, best_state
    if _torch_device.type == 'cuda':
        torch.cuda.empty_cache()                                                                                                                                                        
    gc.collect()
                                                                                                                                                                                        
    return {                                                                                                                                                                          
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),                                                                                                                                            
        'mae':  float(mean_absolute_error(y_te, pred)),
        'pred': pred,                                                                                                                                                                   
    }                                                                                                                                                                                 
                                                                                                                                                                                        

results_mlp = []                                                                                                                                                                        
preds_mlp   = {}                                                                                                                                                                      

for target in TARGET_COLS:
    y_tr_full = df_train[target].to_numpy(dtype=np.float32)
    y_te      = df_test[target].to_numpy(dtype=np.float32)
    global_med = float(np.median(y_tr_full))                                                                                                                                            
                                                                                                                                                                                        
    port_enc_tr, port_enc_te = port_median_encode(                                                                                                                                      
        train_port, y_tr_full, test_port, global_med                                                                                                                                    
    )                                                                                                                                                                                   

    print(f'\n── {target}: gravando memmap...')                                                                                                                                         
    mm, mm_path = build_mmap(X_train_base, port_enc_tr, f'mlp_{target}')                                                                                                              
    sc          = scaler_from_mmap(mm)                                                                                                                                                  
    sm, sm_path = build_scaled_mmap(mm, sc, f'mlp_{target}')                                                                                                                            
    del mm                                                                                                                                                                              
    gc.collect()                                                                                                                                                                        
                                                                                                                                                                                        
    X_te_m      = np.column_stack([X_test_base, port_enc_te])                                                                                                                           
    X_te_scaled = sc.transform(X_te_m).astype(np.float32)
                                                                                                                                                                                        
    seed_preds = []                                                                                                                                                                   
    for seed in range(N_SEEDS):
        row  = train_mlp_one(sm, y_tr_full, X_te_scaled, y_te, seed)                                                                                                                    
        pred = row.pop('pred')                                                                                                                                                          
                                                                                                                                                                                        
        np.save(PREDS_DIR / f'mlp_{target}_seed{seed}.npy', pred)                                                                                                                       
                                                                                                                                                                                    
        seed_preds.append(pred)                                                                                                                                                         
        row.update({'model': 'mlp', 'target': target, 'seed': seed})                                                                                                                  
        results_mlp.append(row)
        print(f'  seed={seed:2d}  RMSE={row["rmse"]:.3f}  R²={row["r2"]:.4f}')                                                                                                          
                                                                                                                                                                                        
    preds_mlp[target] = np.stack(seed_preds)                                                                                                                                            
    rm_mmap(mm_path, sm_path)                                                                                                                                                           
    gc.collect()                                                                                                                                                                        

df_mlp = pd.DataFrame(results_mlp)                                                                                                                                                      
df_mlp.to_csv(METRICS_DIR / 'results_mlp.csv', index=False)                                                                                                                           
print('\nResultados MLP salvos.')                                                                                                                                                       
df_mlp.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)

MLP device: cuda

── TOperacao: gravando memmap...
  seed= 0  RMSE=19.643  R²=0.7089
  seed= 1  RMSE=20.138  R²=0.6940
  seed= 2  RMSE=20.784  R²=0.6741
  seed= 3  RMSE=20.623  R²=0.6791
  seed= 4  RMSE=19.651  R²=0.7086
  seed= 5  RMSE=20.706  R²=0.6765
  seed= 6  RMSE=20.508  R²=0.6827
  seed= 7  RMSE=21.336  R²=0.6565
  seed= 8  RMSE=20.847  R²=0.6721
  seed= 9  RMSE=20.348  R²=0.6876
  seed=10  RMSE=21.158  R²=0.6622
  seed=11  RMSE=20.472  R²=0.6838
  seed=12  RMSE=19.583  R²=0.7107
  seed=13  RMSE=21.137  R²=0.6629
  seed=14  RMSE=21.213  R²=0.6605
  seed=15  RMSE=20.617  R²=0.6793
  seed=16  RMSE=20.749  R²=0.6752
  seed=17  RMSE=20.764  R²=0.6747
  seed=18  RMSE=20.667  R²=0.6777
  seed=19  RMSE=20.402  R²=0.6859

── TAtracado: gravando memmap...
  seed= 0  RMSE=23.438  R²=0.6601
  seed= 1  RMSE=21.986  R²=0.7009
  seed= 2  RMSE=22.891  R²=0.6758
  seed= 3  RMSE=24.208  R²=0.6374
  seed= 4  RMSE=22.973  R²=0.6735
  seed= 5  RMSE=22.570  R²=0.6848
  seed= 6  RMSE=22.114  R²=0.69

rmse              r2              mae        
              mean     std    mean     std     mean     std
target                                                     
TAtracado  23.0851  0.6274  0.6701  0.0180  11.7939  0.3811
TOperacao  20.5673  0.5052  0.6806  0.0156   8.5329  0.2702

## 6. Random Forest

## 9. LightGBM

In [15]:
import lightgbm as lgb

def train_lgbm_one(X_tr, y_tr, X_val, y_val, X_te, y_te, seed):
    model = lgb.LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=127,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=seed,
        device='gpu',
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)],
    )
    pred = model.predict(X_te)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),
        'mae':  float(mean_absolute_error(y_te, pred)),
        'best_iter': int(model.best_iteration_),
        'pred': pred,
        'model_obj': model,
    }


results_lgbm = []
preds_lgbm   = {}

for target in TARGET_COLS:
    y_core_l = df_train[target].to_numpy(dtype=np.float32)[core_idx]
    y_val_l  = df_train[target].to_numpy(dtype=np.float32)[val_idx]
    y_te     = df_test[target].to_numpy(dtype=np.float32)

    global_med = float(np.median(y_core_l))
    port_enc_core, port_enc_test = port_median_encode(
        port_core, y_core_l, test_port, global_med
    )
    port_enc_val_l = pd.Series(port_val).map(
        pd.Series(y_core_l, index=port_core).groupby(level=0).median().to_dict()
    ).fillna(global_med).to_numpy(dtype=np.float32)

    X_tr_l  = np.column_stack([X_core,      port_enc_core])
    X_vl_l  = np.column_stack([X_val,       port_enc_val_l])
    X_te_l  = np.column_stack([X_test_base, port_enc_test])

    seed_preds = []
    print(f'\n── {target} ──')
    for seed in range(N_SEEDS):
        try:
            row = train_lgbm_one(X_tr_l, y_core_l, X_vl_l, y_val_l, X_te_l, y_te, seed)
        except Exception as e:
            # fallback CPU se GPU falhar
            model_cpu = lgb.LGBMRegressor(
                n_estimators=3000, learning_rate=0.05, num_leaves=127,
                min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                n_jobs=-1, random_state=seed,
            )
            model_cpu.fit(
                X_tr_l, y_core_l,
                eval_set=[(X_vl_l, y_val_l)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)],
            )
            pred = model_cpu.predict(X_te_l)
            row = {
                'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
                'r2':   float(r2_score(y_te, pred)),
                'mae':  float(mean_absolute_error(y_te, pred)),
                'best_iter': int(model_cpu.best_iteration_),
                'pred': pred,
                'model_obj': model_cpu,
            }

        pred      = row.pop('pred')
        model_obj = row.pop('model_obj')

        np.save(PREDS_DIR / f'lightgbm_{target}_seed{seed}.npy', pred)
        joblib.dump(model_obj, MODELS_DIR / f'lightgbm_{target}_seed{seed}.pkl')

        seed_preds.append(pred)
        row.update({'model': 'lightgbm', 'target': target, 'seed': seed})
        results_lgbm.append(row)
        print(f'  seed={seed:2d}  RMSE={row["rmse"]:.3f}  R²={row["r2"]:.4f}  iter={row["best_iter"]}')
        gc.collect()

    preds_lgbm[target] = np.stack(seed_preds)

df_lgbm = pd.DataFrame(results_lgbm)
df_lgbm.to_csv(METRICS_DIR / 'results_lightgbm.csv', index=False)
print('\nResultados LightGBM salvos.')
df_lgbm.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)



── TOperacao ──
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 249544
[LightGBM] [Info] Number of data points in the train set: 440213, number of used features: 1610
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1187 dense feature groups (498.75 MB) transferred to GPU in 0.129621 secs. 1 sparse feature groups


KeyboardInterrupt: 

In [26]:
from sklearn.ensemble import RandomForestRegressor

def train_rf_one(mm_X, y_tr, X_te, y_te, seed):
    """
    RF com todos os dados via memmap. joblib(max_nbytes=1) força workers a
    compartilharem o memmap via mmap do SO — sem cópia por processo.
    """
    model = RandomForestRegressor(
        n_estimators=200,
        max_features='sqrt',
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=seed,
    )
    with joblib.parallel_backend('loky', max_nbytes=1):
        model.fit(mm_X, y_tr)
    pred = model.predict(X_te)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
        'r2':   float(r2_score(y_te, pred)),
        'mae':  float(mean_absolute_error(y_te, pred)),
        'pred': pred,
        'model_obj': model,
    }


results_rf = []
preds_rf   = {}

for target in TARGET_COLS:
    y_tr_full = df_train[target].to_numpy(dtype=np.float32)
    y_te      = df_test[target].to_numpy(dtype=np.float32)
    global_med = float(np.median(y_tr_full))

    port_enc_tr, port_enc_te = port_median_encode(
        train_port, y_tr_full, test_port, global_med
    )

    print(f'\n── {target}: gravando memmap...')
    mm, mm_path = build_mmap(X_train_base, port_enc_tr, f'rf_{target}')

    X_te_rf = np.column_stack([X_test_base, port_enc_te])

    seed_preds = []
    for seed in range(N_SEEDS):
        row       = train_rf_one(mm, y_tr_full, X_te_rf, y_te, seed)
        pred      = row.pop('pred')
        model_obj = row.pop('model_obj')

        np.save(PREDS_DIR / f'random_forest_{target}_seed{seed}.npy', pred)
        joblib.dump(model_obj, MODELS_DIR / f'random_forest_{target}_seed{seed}.pkl')

        seed_preds.append(pred)
        row.update({'model': 'random_forest', 'target': target, 'seed': seed})
        results_rf.append(row)
        print(f'  seed={seed:2d}  RMSE={row["rmse"]:.3f}  R²={row["r2"]:.4f}')
        del model_obj
        gc.collect()

    preds_rf[target] = np.stack(seed_preds)
    del mm
    rm_mmap(mm_path)
    gc.collect()

df_rf = pd.DataFrame(results_rf)
df_rf.to_csv(METRICS_DIR / 'results_random_forest.csv', index=False)
print('\nResultados RF salvos.')
df_rf.groupby('target')[['rmse','r2','mae']].agg(['mean','std']).round(4)



── TOperacao: gravando memmap...
  seed= 0  RMSE=21.039  R²=0.6660


KeyboardInterrupt: 

## 7. Resultados consolidados e análise estatística

Wilcoxon pareado seed-a-seed (igual ao protocolo do artigo original).
Compara cada arquitetura entre si.

In [ ]:
df_all = pd.concat([df_xgb, df_rf, df_svm, df_knn, df_lr, df_mlp, df_lgbm], ignore_index=True)
df_all.to_csv(METRICS_DIR / 'results_all_models.csv', index=False)

print('=== RMSE médio por modelo e target ===')
pivot = df_all.groupby(['model','target'])['rmse'].mean().unstack('target').round(3)
print(pivot.to_string())

In [ ]:
def cohens_d(a, b):
    diff = a - b
    return float(diff.mean() / (diff.std(ddof=1) + 1e-12))


models  = ['xgboost', 'lightgbm', 'random_forest', 'svm', 'knn', 'linear_regression', 'mlp']
rmse_by = {
    m: {
        t: df_all[(df_all.model==m) & (df_all.target==t)].sort_values('seed')['rmse'].values
        for t in TARGET_COLS
    }
    for m in models
}

print('=== Wilcoxon pareado seed-a-seed ===')
from itertools import combinations
for m_a, m_b in combinations(models, 2):
    print(f'\n{m_a} vs {m_b}:')
    for t in TARGET_COLS:
        a = rmse_by[m_a][t]
        b = rmse_by[m_b][t]
        if len(a) != len(b) or len(a) < 2:
            continue
        stat, p = wilcoxon(a, b)
        d = cohens_d(a, b)
        winner = m_a if a.mean() < b.mean() else m_b
        sig = '*' if p < 0.05 else 'n.s.'
        print(f'  {t:12s}  {m_a}={a.mean():.3f}  {m_b}={b.mean():.3f}  '
              f'p={p:.4f} {sig}  d={d:+.3f}  melhor={winner}')

print(f'\nMétricas em : {METRICS_DIR.resolve()}')
print(f'Predições em: {PREDS_DIR.resolve()}')
print(f'Modelos em  : {MODELS_DIR.resolve()}')